In [27]:
import yfinance as yf

# 1. 미국 시장(US) 객체 생성
us_market = yf.Market("US")

# 미국 시장의 현재 개장 상태 확인
print(f"미국 시장 상태: {us_market.status}")

# 미국 3대 주요 지수(S&P500, 나스닥 등) 요약 정보 확인
import pandas as pd
display(pd.DataFrame(us_market.summary).T.head())

# ==========================================

# 2. 원자재 시장(COMMODITIES) 객체 생성
commodities = yf.Market("COMMODITIES")

# 원자재 가격 요약 (금, 은, 원유 등)
print("\n--- 주요 원자재 가격 동향 ---")
display(pd.DataFrame(commodities.summary).T.head())

미국 시장 상태: {'id': 'us', 'name': 'U.S. markets', 'status': 'closed', 'yfit_market_id': 'us_market', 'close': datetime.datetime(2026, 4, 13, 20, 0, tzinfo=datetime.timezone.utc), 'duration': [{'hrs': '1', 'mins': '58'}], 'message': 'U.S. markets open in 1 hour 58 minutes', 'open': datetime.datetime(2026, 4, 13, 13, 30, tzinfo=datetime.timezone.utc), 'yfit_market_status': 'YFT_MARKET_WILL_OPEN', 'timezone': {'dst': 'true', 'gmtoffset': '-14400', 'short': 'EDT', '$text': 'America/New_York'}, 'tz': datetime.timezone(datetime.timedelta(days=-1, seconds=34560), 'EDT')}


,language,region,quoteType,typeDisp,quoteSourceName,triggerable,customPriceAlertConfidence,contractSymbol,headSymbolAsString,shortName,...,exchangeTimezoneName,exchangeTimezoneShortName,gmtOffSetMilliseconds,esgPopulated,tradeable,cryptoTradeable,hasPrePostMarketData,firstTradeDateMilliseconds,symbol,priceHint
CME,en-US,US,FUTURE,Futures,Delayed Quote,False,NONE,False,RTY=F,Russell 2000 Futures,...,America/New_York,EDT,-14400000,False,False,False,False,1499659200000,RTY=F,NaN
CBT,en-US,US,FUTURE,Futures,Delayed Quote,False,NONE,False,YM=F,Dow Futures,...,America/New_York,EDT,-14400000,False,False,False,False,1017982800000,YM=F,NaN
CXI,en-US,US,INDEX,Index,NaN,False,LOW,NaN,NaN,VIX,...,America/Chicago,CDT,-18000000,False,False,False,False,631267200000,^VIX,2
CMX,en-US,US,FUTURE,Futures,Delayed Quote,False,NONE,False,GC=F,Gold,...,America/New_York,EDT,-14400000,False,False,False,False,967608000000,GC=F,NaN



--- 주요 원자재 가격 동향 ---


,language,region,quoteType,typeDisp,quoteSourceName,triggerable,customPriceAlertConfidence,headSymbolAsString,contractSymbol,shortName,...,hasPrePostMarketData,firstTradeDateMilliseconds,regularMarketChange,regularMarketChangePercent,regularMarketTime,regularMarketPrice,regularMarketPreviousClose,exchange,market,symbol
NYM,en-US,US,FUTURE,Futures,Delayed Quote,False,NONE,BZ=F,False,Brent Crude Oil Last Day Financ,...,False,1185768000000,6.75,7.090336,1776079382,101.95,95.2,NYM,us24_market,BZ=F
CMX,en-US,US,FUTURE,Futures,Delayed Quote,False,NONE,HG=F,False,Copper May 26,...,False,967608000000,-0.0155,-0.263338,1776079395,5.8705,5.886,CMX,us24_market,HG=F


In [28]:
import yfinance as yf
from datetime import datetime, timedelta

# Default init (today + 7 days)
calendar = yf.Calendars()

# Today's events: calendar of 1 day
tomorrow = datetime.now() + timedelta(days=1)
calendar = yf.Calendars(end=tomorrow)

# Default calendar queries - accessing the properties will fetch the data from YF
calendar.earnings_calendar
calendar.ipo_info_calendar
calendar.splits_calendar
calendar.economic_events_calendar

# Manual queries
calendar.get_earnings_calendar()
calendar.get_ipo_info_calendar()
calendar.get_splits_calendar()
calendar.get_economic_events_calendar()

# Earnings calendar custom filters
calendar.get_earnings_calendar(
    market_cap=100_000_000,  # filter out small-cap 
    filter_most_active=True,  # show only actively traded. Uses: `screen(query="MOST_ACTIVES")`
)

# Example of real use case:
# Get inminent unreported earnings events
today = datetime.now()
is_friday = today.weekday() == 4
day_after_tomorrow = today + timedelta(days=4 if is_friday else 2)

calendar = yf.Calendars(today, day_after_tomorrow)
df = calendar.get_earnings_calendar(limit=100)

unreported_df = df[df["Reported EPS"].isnull()]

In [29]:
import yfinance as yf

# 테슬라(TSLA) 객체 생성
tsla = yf.Ticker("TSLA")

# 1. 애널리스트 목표주가 확인 (딕셔너리로 반환됨)
targets = tsla.analyst_price_targets
print("--- 테슬라 애널리스트 목표주가 ---")
print(f"최고 목표가: ${targets.get('high')}")
print(f"평균 목표가: ${targets.get('mean')}")
print(f"최저 목표가: ${targets.get('low')}")

# 2. 다음 분기 및 내년 실적 예상치 (데이터프레임)
# .loc[]를 써서 올해(0y)와 내년(+1y) 예측치만 필터링해 볼 수 있습니다.
estimates = tsla.earnings_estimate
print("\n--- 테슬라 EPS 실적 예상치 ---")
print(estimates)

# 3. 최대 기관 투자자 목록 확인
institutions = tsla.institutional_holders
print("\n--- 주요 기관 투자자 ---")
# 기관 이름(Holder)과 보유 주식 수(Shares)만 추출
print(institutions[['Holder', 'Shares']].head())

--- 테슬라 애널리스트 목표주가 ---
최고 목표가: $600.0
평균 목표가: $415.2966
최저 목표가: $125.0

--- 테슬라 EPS 실적 예상치 ---
            avg      low  high  yearAgoEps  numberOfAnalysts  growth
period                                                              
0q      0.37735  0.21073  0.51     0.27000                25  0.3976
+1q     0.44637  0.22807  0.59     0.40000                24  0.1159
0y      2.04255  1.11582  2.82     1.66000                34  0.2305
+1y     2.77327  1.43809  6.03     2.04255                33  0.3577

--- 주요 기관 투자자 ---
                          Holder     Shares
0             Vanguard Group Inc  258925024
1                 Blackrock Inc.  209563808
2       State Street Corporation  114842934
3  Geode Capital Management, LLC   65700975
4            JPMORGAN CHASE & CO   44591616


In [30]:
import yfinance as yf

# 1. 의도적으로 오타가 섞인 기업명으로 검색 (Nvidya -> Nvidia)
# enable_fuzzy_query=True를 켜서 유사 검색을 허용합니다.
search_result = yf.Search("Nvidya", enable_fuzzy_query=True, max_results=3)

# 2. 검색된 종목(Quotes) 결과 확인
quotes = search_result.quotes

print("--- 종목 검색 결과 ---")
for quote in quotes:
    # 딕셔너리에서 심볼(티커)과 공식 회사명, 주식 거래소(Exchange) 추출
    symbol = quote.get('symbol')
    short_name = quote.get('shortname')
    exchange = quote.get('exchange')
    print(f"티커: {symbol} | 기업명: {short_name} | 거래소: {exchange}")

# 3. 찾은 티커 중 첫 번째(가장 정확한 결과)를 이용하여 Ticker 객체 생성 후 가격 확인
if quotes:
    exact_ticker = quotes[0].get('symbol') # 'NVDA'가 추출됨
    
    # 여기서부터는 이전에 배운 Ticker 클래스의 영역입니다!
    nvda = yf.Ticker(exact_ticker)
    current_price = nvda.fast_info.last_price
    print(f"\n=> 검색된 {exact_ticker}의 현재가: ${current_price:.2f}")

# 4. 관련된 최신 뉴스 확인
print("\n--- 관련 최신 뉴스 ---")
news_list = search_result.news
for article in news_list[:2]: # 2개만 출력
    print(f"- {article.get('title')}")

--- 종목 검색 결과 ---
티커: NVDA | 기업명: NVIDIA Corporation | 거래소: NMS
티커: NVDY | 기업명: YieldMax NVDA Option Income Str | 거래소: PCX
티커: NVDU | 기업명: Direxion Daily NVDA Bull 2X ETF | 거래소: NGM

=> 검색된 NVDA의 현재가: $188.63

--- 관련 최신 뉴스 ---
- With Brain Scan Volume Momentum Accelerating, Firefly Makes History by Building the World’s First Known 200,000+ Standardized EEG/ERP Depository
- Ninepoint Partners Launching Expanded Suite of Single-Stock ETFs


In [31]:
import yfinance as yf
import pandas as pd

# 1. 기존의 Search 객체를 통해 모든 결과 검색
search_result = yf.Search("Gold", max_results=30)
df_all = pd.DataFrame(search_result.quotes)

# 2. 'quoteType' 컬럼을 검사하여 'EQUITY'(개별 주식)만 필터링
if not df_all.empty and 'quoteType' in df_all.columns:
    df_stocks = df_all[df_all['quoteType'] == 'EQUITY']
    print("--- 직접 필터링한 주식 목록 ---")
    print(df_stocks[['symbol', 'shortname', 'exchange']])
else:
    print("검색 결과가 없습니다.")

--- 직접 필터링한 주식 목록 ---
  symbol       shortname exchange
0   GOLD  Gold.com, Inc.      NYQ


In [ ]:
import yfinance as yf
import asyncio

# 1. 수신된 데이터를 처리할 비동기 핸들러 함수
async def my_async_handler(message):
    ticker = message.get('id')
    price = message.get('price')
    if ticker and price:
        print(f"👉 [실시간 체결] {ticker}: ${price:.2f}")

# 2. 비동기 메인 함수 정의
async def main():
    # AsyncWebSocket 객체 생성
    aws = yf.AsyncWebSocket(verbose=False)
    
    try:
        # 비동기로 구독 요청 (await 필수)
        await aws.subscribe(['NVDA'])
        print("엔비디아(NVDA) 실시간 스트리밍 시작...")

        # listen()을 '백그라운드 태스크(Task)'로 실행하여 블로킹 방지
        # 이렇게 하면 데이터를 받으면서도 아래의 while 루프를 동시에 돌릴 수 있습니다.
        listen_task = asyncio.create_task(aws.listen(message_handler=my_async_handler))

        # 메인 프로그램은 다른 작업을 동시에 수행
        while True:
            print("...[백그라운드] 다른 데이터 분석 모델 연산 중...")
            await asyncio.sleep(5)  # 5초 대기

    except asyncio.CancelledError:
        print("\n프로그램 종료 요청을 받았습니다.")
    finally:
        # 종료 시 안전하게 웹소켓 닫기
        await aws.close()
        print("웹소켓 연결이 종료되었습니다.")

# 3. 비동기 이벤트 루프 실행
await main()

엔비디아(NVDA) 실시간 스트리밍 시작...
...[백그라운드] 다른 데이터 분석 모델 연산 중...
👉 [실시간 체결] NVDA: $185.66
👉 [실시간 체결] NVDA: $185.69
👉 [실시간 체결] NVDA: $185.69
👉 [실시간 체결] NVDA: $185.66
...[백그라운드] 다른 데이터 분석 모델 연산 중...
👉 [실시간 체결] NVDA: $185.68
👉 [실시간 체결] NVDA: $185.66
...[백그라운드] 다른 데이터 분석 모델 연산 중...
👉 [실시간 체결] NVDA: $185.66
...[백그라운드] 다른 데이터 분석 모델 연산 중...
👉 [실시간 체결] NVDA: $185.68
👉 [실시간 체결] NVDA: $185.68

프로그램 종료 요청을 받았습니다.
👉 [실시간 체결] NVDA: $185.69


In [2]:
import yfinance as yf

# 1. 'technology' (테크/IT) 섹터 객체 생성
tech_sector = yf.Sector('technology')

# 2. 테크 섹터 내의 하위 산업(Industry) 비중 확인
industries_df = tech_sector.industries
print("--- 테크 섹터 내 하위 산업 구조 ---")
# 가중치(Market Weight) 순으로 정렬되어 나옵니다.
print(industries_df.head())

# 3. 테크 섹터를 이끄는 상위 5개 대장주(Top Companies) 추출
top_tech_companies = tech_sector.top_companies
print("\n--- 테크 섹터 Top 5 기업 ---")
print(top_tech_companies.reset_index()[['symbol', 'name', 'market weight']].head(5))

# 4. 테크 섹터에 투자하는 상위 ETF 확인 (딕셔너리 반환)
top_etfs = tech_sector.top_etfs
print("\n--- 대표 테크 ETF 목록 ---")
for etf_symbol, etf_name in list(top_etfs.items())[:3]: # 3개만 출력
    print(f"- {etf_symbol}: {etf_name}")

--- 테크 섹터 내 하위 산업 구조 ---
                                                                  name  \
key                                                                      
semiconductors                                          Semiconductors   
software-infrastructure                      Software - Infrastructure   
consumer-electronics                              Consumer Electronics   
software-application                            Software - Application   
semiconductor-equipment-materials  Semiconductor Equipment & Materials   

                                        symbol  market weight  
key                                                            
semiconductors                     ^YH31130020       0.381367  
software-infrastructure            ^YH31110030       0.196527  
consumer-electronics               ^YH31120030       0.170482  
software-application               ^YH31110020       0.068780  
semiconductor-equipment-materials  ^YH31130010       0.047999  

--- 테크 

In [6]:
import yfinance as yf

# 1. '반도체' 산업 객체 생성
# (앞서 Sector의 industries 목록에서 확인한 key 값을 넣습니다)
semi_industry = yf.Industry('semiconductors')

# 2. 소속 정보 확인
print(f"--- {semi_industry.name} 산업 요약 ---")
print(f"소속 섹터: {semi_industry.sector_name}")

# 3. 반도체 대장주 Top 3 (안정성)
print("\n[1] 반도체 시가총액 Top 3 대장주")
top_cos = semi_industry.top_companies
# 결과가 DataFrame이므로 필요한 컬럼만 추출
print(top_cos.reset_index()[['symbol', 'name', 'market weight']].head(3))

# 4. 반도체 고성장주 Top 3 (성장성)
print("\n[2] 업계 평균을 상회하는 고성장 반도체 기업")
top_growth = semi_industry.top_growth_companies
if top_growth is not None:
    print(top_growth.reset_index()[['symbol', 'name', 'growth estimate', 'ytd return']].head(3))

# 5. 최근 퍼포먼스가 좋은 반도체 기업 Top 3 (모멘텀)
print("\n[3] 최근 주가 흐름이 가장 좋은 반도체 기업")
top_perf = semi_industry.top_performing_companies
if top_perf is not None:
    print(top_perf.reset_index()[['symbol', 'name', 'ytd return', 'last price']].head(3))

--- Semiconductors 산업 요약 ---
소속 섹터: Technology

[1] 반도체 시가총액 Top 3 대장주
  symbol                     name  market weight
0   NVDA       NVIDIA Corporation       0.536931
1   AVGO            Broadcom Inc.       0.206186
2     MU  Micron Technology, Inc.       0.055515

[2] 업계 평균을 상회하는 고성장 반도체 기업
  symbol                     name  growth estimate  ytd return
0     MU  Micron Technology, Inc.         5.846154      0.4736
1   SMTC      Semtech Corporation         3.857143      0.1565
2   SITM       SiTime Corporation         3.833333      0.1901

[3] 최근 주가 흐름이 가장 좋은 반도체 기업
  symbol                    name  ytd return  last price
0   QUIK  QuickLogic Corporation      0.8486       11.11
1   LASR            nLIGHT, Inc.      0.7427       65.37
2   INTC       Intel Corporation      0.6905       62.38


In [12]:
import yfinance as yf
from yfinance import FundQuery
import pandas as pd

# 1. 쿼리 정의
query = FundQuery('and', [
    FundQuery('eq',    ['categoryname',                'Large Growth']),
    FundQuery('is-in', ['performanceratingoverall',    4, 5]),
    FundQuery('lt',    ['initialinvestment',           100001]),
    FundQuery('lt',    ['annualreturnnavy1categoryrank', 50]),
    FundQuery('eq',    ['exchange',                    'NAS'])
])

# 2. 스크리닝 실행
# size: 한 번에 반환할 최대 개수 (최대 250)
result = yf.screen(query, size=20)
print(f"조건 충족 펀드: 총 {result['total']}개 발견 / {result['count']}개 반환\n")

# 3. 결과를 DataFrame으로 변환
df = pd.DataFrame(result['quotes'])

# 4. 주요 컬럼만 추출해서 출력
cols = ['symbol', 'shortName', 'exchange',
        'regularMarketPrice',         # 현재 NAV(가격)
        'ytdReturn',                  # 연초 대비 수익률
        'trailingThreeMonthReturns',  # 최근 3개월 수익률
        'netAssets',                  # 순자산 규모
        'netExpenseRatio',            # 운용 보수율
        'dividendYield']              # 배당 수익률

df_result = df[cols].copy()

# 5. 가독성을 위한 포맷팅
df_result.columns = ['티커', '펀드명', '거래소',
                     '현재가($)', 'YTD수익률', '3개월수익률',
                     '순자산($)', '운용보수율', '배당수익률']

# 수익률 퍼센트 표시
for col in ['YTD수익률', '3개월수익률', '운용보수율', '배당수익률']:
    df_result[col] = df_result[col].apply(
        lambda x: f"{x*100:.2f}%" if pd.notna(x) else '-'
    )

# 순자산 단위 변환 (억 달러)
df_result['순자산($)'] = df_result['순자산($)'].apply(
    lambda x: f"${x/1e9:.1f}B" if pd.notna(x) else '-'
)

print(df_result.to_string(index=False))
import yfinance as yf
from yfinance import FundQuery
import pandas as pd

# 1. 쿼리 정의
query = FundQuery('and', [
    FundQuery('eq',    ['categoryname',                'Large Growth']),
    FundQuery('is-in', ['performanceratingoverall',    4, 5]),
    FundQuery('lt',    ['initialinvestment',           100001]),
    FundQuery('lt',    ['annualreturnnavy1categoryrank', 50]),
    FundQuery('eq',    ['exchange',                    'NAS'])
])

# 2. 스크리닝 실행
# size: 한 번에 반환할 최대 개수 (최대 250)
result = yf.screen(query, size=20)
print(f"조건 충족 펀드: 총 {result['total']}개 발견 / {result['count']}개 반환\n")

# 3. 결과를 DataFrame으로 변환
df = pd.DataFrame(result['quotes'])

# 4. 주요 컬럼만 추출해서 출력
cols = ['symbol', 'shortName', 'exchange',
        'regularMarketPrice',         # 현재 NAV(가격)
        'ytdReturn',                  # 연초 대비 수익률
        'trailingThreeMonthReturns',  # 최근 3개월 수익률
        'netAssets',                  # 순자산 규모
        'netExpenseRatio',            # 운용 보수율
        'dividendYield']              # 배당 수익률

df_result = df[cols].copy()

# 5. 가독성을 위한 포맷팅
df_result.columns = ['티커', '펀드명', '거래소',
                     '현재가($)', 'YTD수익률', '3개월수익률',
                     '순자산($)', '운용보수율', '배당수익률']

# 수익률 퍼센트 표시
for col in ['YTD수익률', '3개월수익률', '운용보수율', '배당수익률']:
    df_result[col] = df_result[col].apply(
        lambda x: f"{x*100:.2f}%" if pd.notna(x) else '-'
    )

# 순자산 단위 변환 (억 달러)
df_result['순자산($)'] = df_result['순자산($)'].apply(
    lambda x: f"${x/1e9:.1f}B" if pd.notna(x) else '-'
)

display(df_result)


조건 충족 펀드: 총 163개 발견 / 20개 반환

   티커                             펀드명 거래소  현재가($)    YTD수익률    3개월수익률  순자산($)   운용보수율  배당수익률
VRGWX Vanguard Russell 1000 Growth In NAS  883.37  -978.43%  -978.43%  $44.9B   5.00% 52.00%
VMGAX Vanguard Mega Cap 300 Growth In NAS  770.08 -1087.53% -1087.53%  $27.9B   5.00% 40.00%
VIGRX Vanguard Index Trust Growth Ind NAS  237.49 -1042.04% -1042.04% $317.9B  17.00% 32.00%
VIGIX Vanguard Index Trust - Growth I NAS  237.41 -1038.66% -1038.66% $317.9B   3.00% 45.00%
VIGAX Vanguard Growth Index Fd Admira NAS  237.40 -1039.50% -1039.50% $317.9B   5.00% 44.00%
VCNIX VALIC Company I Nasdaq 100 Inde NAS   24.67  -591.21%  -591.21%   $1.0B  42.00% 36.00%
USNQX Victory NASDAQ-100 Index Fund", NAS   60.21  -593.49%  -593.49%   $7.7B  42.00% 19.00%
URNQX Victory Nasdaq 100 Index Fund R NAS   60.26  -589.89%  -589.89%   $7.7B  29.00% 31.00%
UPDDX    Upright Growth & Income Fund NAS   27.42   -27.22%   -27.22%   $0.0B 253.00%  0.00%
UINQX Victory Nasdaq 100 Index Fund I NA

,티커,펀드명,거래소,현재가($),YTD수익률,3개월수익률,순자산($),운용보수율,배당수익률
0,VRGWX,Vanguard Russell 1000 Growth In,NAS,883.37,-978.43%,-978.43%,$44.9B,5.00%,52.00%
1,VMGAX,Vanguard Mega Cap 300 Growth In,NAS,770.08,-1087.53%,-1087.53%,$27.9B,5.00%,40.00%
2,VIGRX,Vanguard Index Trust Growth Ind,NAS,237.49,-1042.04%,-1042.04%,$317.9B,17.00%,32.00%
3,VIGIX,Vanguard Index Trust - Growth I,NAS,237.41,-1038.66%,-1038.66%,$317.9B,3.00%,45.00%
4,VIGAX,Vanguard Growth Index Fd Admira,NAS,237.40,-1039.50%,-1039.50%,$317.9B,5.00%,44.00%
5,VCNIX,VALIC Company I Nasdaq 100 Inde,NAS,24.67,-591.21%,-591.21%,$1.0B,42.00%,36.00%
6,USNQX,"Victory NASDAQ-100 Index Fund"",",NAS,60.21,-593.49%,-593.49%,$7.7B,42.00%,19.00%
7,URNQX,Victory Nasdaq 100 Index Fund R,NAS,60.26,-589.89%,-589.89%,$7.7B,29.00%,31.00%
8,UPDDX,Upright Growth & Income Fund,NAS,27.42,-27.22%,-27.22%,$0.0B,253.00%,0.00%
9,UINQX,Victory Nasdaq 100 Index Fund I,NAS,60.29,-592.70%,-592.70%,$7.7B,43.00%,18.00%


In [13]:
import yfinance as yf

# 1. 디버그 모드 활성화 (내부 통신 로그 출력 켜기)
yf.enable_debug_mode()

# 2. 고의로 존재하지 않는 이상한 티커(종목)를 요청해 봅니다.
print("--- 데이터 요청 시작 ---")
ticker = yf.Ticker("INVALID_TICKER_123")
data = ticker.history(period="1mo")

# 터미널 창을 확인해 보면, 어떤 URL을 호출했고 왜 실패했는지 
# (예: 404 Not Found) 상세한 에러 트레이스가 빨간/흰색 글씨로 쏟아집니다.

# (참고) 디버깅이 끝나고 로그가 너무 시끄럽다면 다시 끌 수 있습니다.
# yf.config.debug.logging = False

c:\Users\gorhk\최종 프로젝트\finance_project\.venv\Lib\site-packages\yfinance\utils.py:167: DeprecationWarning: enable_debug_mode() is replaced by: yf.config.debug.logging = True (or False to disable)
  warnings.warn("enable_debug_mode() is replaced by: yf.config.debug.logging = True (or False to disable)", DeprecationWarning)


--- 데이터 요청 시작 ---


ERROR    HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: INVALID_TICKER_123"}}}
ERROR    $INVALID_TICKER_123: possibly delisted; no price data found  (period=1mo) (Yahoo error = "No data found, symbol may be delisted")


In [14]:
import yfinance as yf
import tempfile
from pathlib import Path

# 1. 시스템의 기본 임시 폴더(Temporary Directory) 경로를 안전하게 가져옵니다.
# (리눅스/클라우드 환경에서는 보통 '/tmp'로 자동 지정됩니다.)
temp_dir = tempfile.gettempdir()

# 2. yfinance 시간대 캐시 저장 위치를 해당 임시 폴더로 변경합니다.
yf.set_tz_cache_location(temp_dir)
print(f"시간대 캐시 폴더가 다음으로 설정되었습니다: {temp_dir}")

# 3. 이제 권한 에러 걱정 없이 클라우드 환경에서도 데이터를 마음껏 수집합니다!
print("\n--- 데이터 수집 시작 ---")
msft = yf.Ticker("MSFT")
history = msft.history(period="1mo")
print(history[['Close']].head())

시간대 캐시 폴더가 다음으로 설정되었습니다: C:\Users\gorhk\AppData\Local\Temp

--- 데이터 수집 시작 ---
                                Close
Date                                 
2026-03-13 00:00:00-04:00  395.549988
2026-03-16 00:00:00-04:00  399.950012
2026-03-17 00:00:00-04:00  399.410004
2026-03-18 00:00:00-04:00  391.790009
2026-03-19 00:00:00-04:00  389.019989
